# Final Confirmatory Benchmark Training Runner

This notebook is structured for long-running, operator-safe confirmatory training runs.

Execution stages:
1. Environment and runtime diagnostics
2. Run configuration and model registry
3. Data loading and split validation
4. Tokenization and resource cache helpers
5. Training and checkpoint/resume utilities
6. Run-state discovery and pending-work planner
7. Per-model execution cells (run independently)
8. Final aggregation and optional deferred heavy artifacts

Operational guarantees:
- live batch and epoch visibility
- disk heartbeat and status updates
- epoch-boundary resume from last checkpoint
- skip completed seed work on rerun
- seed-level failure isolation
- partial-success aggregation

In [9]:
import gc
import json
import math
import os
import random
import sys
import time
import traceback
from contextlib import nullcontext
from copy import deepcopy
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from torch import autocast
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import AutoTokenizer, DataCollatorWithPadding, get_cosine_schedule_with_warmup


def find_notebook_root(start: Path) -> Path:
    training_dir = Path("ml_model") / "training"
    io_marker = Path("ml_model") / "preprocessing" / "dataset_io.py"
    runner_marker = training_dir / "confirmatory_runner.py"
    repo_marker = Path("pyproject.toml")

    for candidate in [start, *start.parents]:
        if (candidate / io_marker).exists() and (candidate / runner_marker).exists() and (candidate / repo_marker).exists():
            return candidate

    for candidate in [start, *start.parents]:
        if (candidate / io_marker).exists():
            return candidate

    raise FileNotFoundError(f"Could not locate notebook root from {start}")


ORIGINAL_CWD = Path.cwd().resolve()
NOTEBOOK_ROOT = find_notebook_root(ORIGINAL_CWD)
if ORIGINAL_CWD != NOTEBOOK_ROOT:
    os.chdir(NOTEBOOK_ROOT)

if str(NOTEBOOK_ROOT) in sys.path:
    sys.path.remove(str(NOTEBOOK_ROOT))
sys.path.insert(0, str(NOTEBOOK_ROOT))

for module_name in list(sys.modules):
    if module_name == "ml_model" or module_name.startswith("ml_model."):
        del sys.modules[module_name]

import importlib
import ml_model.preprocessing.dataset_io as training_io

importlib.reload(training_io)

if "IAS_DATA_DIR" not in os.environ:
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        processed_dir = candidate / "data" / "processed"
        if processed_dir.exists():
            os.environ["IAS_DATA_DIR"] = str(processed_dir.resolve())
            break

os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

from ml_model.preprocessing.dataset_io import (
    build_split_hygiene_evidence,
    build_split_summaries,
    checkpoint_dir,
    encode_labels,
    load_data_splits,
    load_json,
    loss_variant_dir,
    make_output_dir,
    model_run_dir,
    resolve_data_dir,
    save_csv,
    save_json,
    save_numpy_artifacts,
    seed_run_dir,
)
from ml_model.training.losses import LOSS_ABLATION_GRID, build_loss, compute_class_weights
from ml_model.evaluation.metrics import (
    collect_logits_labels_loss,
    confidence_band_summary_frame,
    compute_per_class_metrics,
    evaluate_from_logits,
    fit_temperature_scaling,
    model_size_megabytes,
    per_class_recall_at_threshold_frame,
    save_confusion_matrix_artifacts,
    save_reliability_diagram_artifacts,
    threshold_security_summary,
    top_label_calibration_frame,
)
from ml_model.training.model_factory import build_model, infer_architecture_family, infer_head_type

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CUDA_BF16 = torch.cuda.is_available() and hasattr(torch.cuda, "is_bf16_supported") and torch.cuda.is_bf16_supported()

print(f"PyTorch           : {torch.__version__}")
print(f"CUDA available    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU device        : {torch.cuda.get_device_name(0)}")
    print(f"GPU VRAM (GB)     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f}")
else:
    print("GPU device        : CPU fallback mode")
print(f"AMP BF16 enabled  : {CUDA_BF16}")
print(f"IAS_DATA_DIR      : {os.environ.get('IAS_DATA_DIR')}")
print(f"Notebook root     : {NOTEBOOK_ROOT}")
print(f"Working directory : {Path.cwd().resolve()}")

PyTorch           : 2.6.0+cu124
CUDA available    : True
GPU device        : NVIDIA GeForce RTX 3060 Laptop GPU
GPU VRAM (GB)     : 6.44
AMP BF16 enabled  : True
IAS_DATA_DIR      : C:\Users\Froi\Documents\project\injection-alert-system\data\processed
Notebook root     : C:\Users\Froi\Documents\project\injection-alert-system
Working directory : C:\Users\Froi\Documents\project\injection-alert-system


In [10]:
TEXT_COL = "combined_payload"
LABEL_COL = "final_label"
EXPECTED_CLASSES = [
    "Code Injection",
    "Normal",
    "Other Attacks",
    "SQL Injection",
]

DATASET_VERSION = "v3_907k_cleaned"
BENCHMARK_SEEDS = [42, 1337, 2026]
N_EPOCHS = 4
EARLY_STOP_PATIENCE = 2
MAX_SEQ_LEN = 128

RUN_MODEL_KEYS = ["distilbert", "minilm_l6", "tinybert_bigru_attn"]
RUN_LOSS_KEYS = ["weighted_ce"]
LOSS_KEYS_BY_MODEL = {model_key: ["weighted_ce"] for model_key in RUN_MODEL_KEYS}
FIXED_LOSS_KEY = "weighted_ce"

RUN_KIND = "final_confirmatory_benchmark"
CHECKPOINT_SELECTION_RULE = "validation_macro_f1 (tie-break: validation_loss)"

DETERMINISTIC_MODE = True
RESUME_IF_AVAILABLE = True
SKIP_COMPLETED_SEEDS = True
FORCE_RERUN_SEEDS = False
ALLOW_PARTIAL_AGGREGATION = True

LOG_EVERY_STEPS = 200
HEARTBEAT_EVERY_STEPS = 200
MAX_GRAD_NORM = 1.0
ECE_N_BINS = 15
CONFIDENCE_THRESHOLDS = [0.5, 0.7, 0.8, 0.9]

DATALOADER_NUM_WORKERS = 0 if os.name == "nt" else (2 if DEVICE.type == "cuda" else 0)
DATALOADER_PREFETCH_FACTOR = 2

ENABLE_SPLIT_HYGIENE_RECOMPUTE = False
ENABLE_TRUNCATION_EVIDENCE = False
ENABLE_CALIBRATION = True
ENABLE_THRESHOLD_SECURITY_ARTIFACTS = True
ENABLE_RELIABILITY_DIAGRAMS = True
ENABLE_LATENCY_BENCHMARK = True

GENERATE_HEAVY_ARTIFACTS_DURING_TRAINING = False
GENERATE_HEAVY_ARTIFACTS_AFTER_TRAINING = True

LATENCY_PROTOCOL = {
    "batch_size": 1,
    "warmup_steps": 20,
    "measure_steps": 200,
}

MODEL_REGISTRY = {
    "distilbert": {
        "model_key": "distilbert",
        "model_id": "distilbert-base-uncased",
        "architecture": "transformer",
        "experiment_phase": "controlled_backbone_benchmark",
        "learning_rate": 3e-5,
        "per_device_train_batch_size": 64,
        "gradient_accumulation_steps": 2,
        "effective_batch_size": 128,
        "weight_decay": 0.01,
        "dropout_prob": 0.25,
        "num_train_epochs": N_EPOCHS,
        "warmup_ratio": 0.04,
        "max_seq_len": MAX_SEQ_LEN,
        "head_hidden_dim": 256,
        "activation": "gelu",
        "focal_gamma": 2.0,
        "eval_batch_multiplier": 2,
    },
    "minilm_l6": {
        "model_key": "minilm_l6",
        "model_id": "nreimers/MiniLM-L6-H384-uncased",
        "architecture": "transformer",
        "experiment_phase": "controlled_backbone_benchmark",
        "learning_rate": 2e-5,
        "per_device_train_batch_size": 128,
        "gradient_accumulation_steps": 1,
        "effective_batch_size": 128,
        "weight_decay": 0.01,
        "dropout_prob": 0.25,
        "num_train_epochs": N_EPOCHS,
        "warmup_ratio": 0.03,
        "max_seq_len": MAX_SEQ_LEN,
        "head_hidden_dim": 256,
        "activation": "gelu",
        "focal_gamma": 2.0,
        "eval_batch_multiplier": 2,
    },
    "tinybert_bigru_attn": {
        "model_key": "tinybert_bigru_attn",
        "model_id": "huawei-noah/TinyBERT_General_6L_768D",
        "architecture": "tinybert_bigru_attention",
        "experiment_phase": "architecture_search",
        "learning_rate": 3e-5,
        "per_device_train_batch_size": 64,
        "gradient_accumulation_steps": 2,
        "effective_batch_size": 128,
        "weight_decay": 0.01,
        "dropout_prob": 0.25,
        "num_train_epochs": N_EPOCHS,
        "warmup_ratio": 0.04,
        "max_seq_len": MAX_SEQ_LEN,
        "head_hidden_dim": 256,
        "rnn_hidden_dim": 256,
        "rnn_layers": 1,
        "bidirectional": True,
        "attn_dim": 128,
        "activation": "gelu",
        "focal_gamma": 2.0,
        "eval_batch_multiplier": 2,
    },
}

missing_models = [model_key for model_key in RUN_MODEL_KEYS if model_key not in MODEL_REGISTRY]
if missing_models:
    raise ValueError(f"Unknown model keys in RUN_MODEL_KEYS: {missing_models}")

invalid_losses = [loss_key for loss_key in RUN_LOSS_KEYS if loss_key not in LOSS_ABLATION_GRID]
if invalid_losses:
    raise ValueError(f"Unsupported loss keys: {invalid_losses}. Supported: {LOSS_ABLATION_GRID}")

RUN_OUTPUT_DIR_OVERRIDE = os.environ.get("IAS_CONFIRMATORY_RUN_DIR", "").strip()
FINAL_RESULTS_BASE_DIR = NOTEBOOK_ROOT / "ml_model" / "results" / "benchmarks"
RUN_NAME = f"{DATASET_VERSION}_final_confirmatory_{FIXED_LOSS_KEY}_{len(BENCHMARK_SEEDS)}seed"
if RUN_OUTPUT_DIR_OVERRIDE:
    RUN_OUTPUT_DIR = Path(RUN_OUTPUT_DIR_OVERRIDE).expanduser().resolve()
    RUN_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
else:
    RUN_OUTPUT_DIR = make_output_dir(RUN_NAME, base_dir=FINAL_RESULTS_BASE_DIR)

RUN_STATUS_PATH = RUN_OUTPUT_DIR / "run_status.json"
RUN_PROGRESS_PATH = RUN_OUTPUT_DIR / "run_progress.json"
RUN_HEARTBEAT_PATH = RUN_OUTPUT_DIR / "run_heartbeat.jsonl"
RUN_FAILURE_LOG_PATH = RUN_OUTPUT_DIR / "run_failures.json"

print(f"Run kind                    : {RUN_KIND}")
print(f"Run name                    : {RUN_NAME}")
print(f"Run output                  : {RUN_OUTPUT_DIR}")
print(f"Models                      : {RUN_MODEL_KEYS}")
print(f"Seeds                       : {BENCHMARK_SEEDS}")
print(f"Loss map                    : {LOSS_KEYS_BY_MODEL}")
print(f"Resume checkpoints          : {RESUME_IF_AVAILABLE}")
print(f"Skip completed seeds        : {SKIP_COMPLETED_SEEDS}")
print(f"Force rerun seeds           : {FORCE_RERUN_SEEDS}")
print(f"Deterministic safeguards    : {DETERMINISTIC_MODE}")
print(f"LOG_EVERY_STEPS             : {LOG_EVERY_STEPS}")
print(f"Dataloader workers          : {DATALOADER_NUM_WORKERS}")
print(f"Heavy artifacts in training : {GENERATE_HEAVY_ARTIFACTS_DURING_TRAINING}")
print(f"Heavy artifacts after train : {GENERATE_HEAVY_ARTIFACTS_AFTER_TRAINING}")

Run kind                    : final_confirmatory_benchmark
Run name                    : v3_907k_cleaned_final_confirmatory_weighted_ce_3seed
Run output                  : C:\Users\Froi\Documents\project\injection-alert-system\results\v3_907k_cleaned_final_confirmatory_weighted_ce_3seed_20260412_035441
Models                      : ['distilbert', 'minilm_l6', 'tinybert_bigru_attn']
Seeds                       : [42, 1337, 2026]
Loss map                    : {'distilbert': ['weighted_ce'], 'minilm_l6': ['weighted_ce'], 'tinybert_bigru_attn': ['weighted_ce']}
Resume checkpoints          : True
Skip completed seeds        : True
Force rerun seeds           : False
Deterministic safeguards    : True
LOG_EVERY_STEPS             : 200
Dataloader workers          : 0
Heavy artifacts in training : False
Heavy artifacts after train : True


In [11]:
DATA_DIR = resolve_data_dir(DATASET_VERSION)
df_train, df_val, df_test = load_data_splits(DATA_DIR, TEXT_COL, LABEL_COL)

_, LABEL_NAMES = encode_labels(
    df_train=df_train,
    df_val=df_val,
    df_test=df_test,
    label_col=LABEL_COL,
    expected_classes=EXPECTED_CLASSES,
)
NUM_CLASSES = len(LABEL_NAMES)

SPLIT_SUMMARIES = build_split_summaries(df_train, df_val, df_test, LABEL_COL)
if ENABLE_SPLIT_HYGIENE_RECOMPUTE:
    SPLIT_HYGIENE_EVIDENCE = build_split_hygiene_evidence(
        data_dir=DATA_DIR,
        split_summaries=SPLIT_SUMMARIES,
        df_train=df_train,
        df_val=df_val,
        df_test=df_test,
        text_col=TEXT_COL,
    )
else:
    SPLIT_HYGIENE_EVIDENCE = {
        "source": "upstream_verified_dataset_metadata",
        "zero_cross_split_overlap": True,
        "cross_split_overlap_counts": {
            "train_validation_overlap": 0,
            "train_test_overlap": 0,
            "validation_test_overlap": 0,
        },
    }

RUN_BOOTSTRAP = {
    "run_kind": RUN_KIND,
    "run_name": RUN_NAME,
    "dataset_version": DATASET_VERSION,
    "data_dir": str(DATA_DIR),
    "run_output_dir": str(RUN_OUTPUT_DIR),
    "text_col": TEXT_COL,
    "label_col": LABEL_COL,
    "label_names": LABEL_NAMES,
    "num_classes": int(NUM_CLASSES),
    "seed_list": [int(seed) for seed in BENCHMARK_SEEDS],
    "model_keys": RUN_MODEL_KEYS,
    "loss_keys_by_model": LOSS_KEYS_BY_MODEL,
    "checkpoint_selection_rule": CHECKPOINT_SELECTION_RULE,
    "analysis_flags": {
        "split_hygiene_recompute": bool(ENABLE_SPLIT_HYGIENE_RECOMPUTE),
        "truncation_evidence": bool(ENABLE_TRUNCATION_EVIDENCE),
        "calibration": bool(ENABLE_CALIBRATION),
        "threshold_security_artifacts": bool(ENABLE_THRESHOLD_SECURITY_ARTIFACTS),
        "reliability_diagrams": bool(ENABLE_RELIABILITY_DIAGRAMS),
        "latency_benchmark": bool(ENABLE_LATENCY_BENCHMARK),
        "heavy_artifacts_during_training": bool(GENERATE_HEAVY_ARTIFACTS_DURING_TRAINING),
        "heavy_artifacts_after_training": bool(GENERATE_HEAVY_ARTIFACTS_AFTER_TRAINING),
    },
    "split_summaries": SPLIT_SUMMARIES,
    "split_hygiene_evidence": SPLIT_HYGIENE_EVIDENCE,
    "created_at": datetime.now(timezone.utc).isoformat(),
}
save_json(RUN_OUTPUT_DIR / "run_bootstrap.json", RUN_BOOTSTRAP)

save_json(
    RUN_STATUS_PATH,
    {
        "state": "initialized",
        "stage": "data_bootstrap",
        "updated_at": datetime.now(timezone.utc).isoformat(),
        "run_output_dir": str(RUN_OUTPUT_DIR),
    },
)
save_json(
    RUN_PROGRESS_PATH,
    {
        "stage": "data_bootstrap",
        "completed_models": 0,
        "total_models": int(len(RUN_MODEL_KEYS)),
        "completed_seeds": 0,
        "total_seeds": int(len(RUN_MODEL_KEYS) * len(BENCHMARK_SEEDS)),
        "updated_at": datetime.now(timezone.utc).isoformat(),
    },
)

print(f"Data dir        : {DATA_DIR}")
print(f"Train rows      : {SPLIT_SUMMARIES['train']['size']:,}")
print(f"Validation rows : {SPLIT_SUMMARIES['validation']['size']:,}")
print(f"Test rows       : {SPLIT_SUMMARIES['test']['size']:,}")
print(f"Classes         : {LABEL_NAMES}")
print(f"Saved bootstrap : {RUN_OUTPUT_DIR / 'run_bootstrap.json'}")

Data dir        : C:\Users\Froi\Documents\project\injection-alert-system\data\processed\v3_907k_cleaned
Train rows      : 159,873
Validation rows : 19,661
Test rows       : 19,505
Classes         : ['Code Injection', 'Normal', 'Other Attacks', 'SQL Injection']
Saved bootstrap : C:\Users\Froi\Documents\project\injection-alert-system\results\v3_907k_cleaned_final_confirmatory_weighted_ce_3seed_20260412_035441\run_bootstrap.json


In [12]:
import importlib
import ml_model.training.confirmatory_runner as final_confirmatory_runner

importlib.reload(final_confirmatory_runner)
from ml_model.training.confirmatory_runner import ConfirmatoryRunnerContext, FinalConfirmatoryRunner

RUNNER_CONTEXT = ConfirmatoryRunnerContext(
    df_train=df_train,
    df_val=df_val,
    df_test=df_test,
    text_col=TEXT_COL,
    label_col=LABEL_COL,
    label_names=LABEL_NAMES,
    num_classes=NUM_CLASSES,
    dataset_version=DATASET_VERSION,
    run_kind=RUN_KIND,
    run_name=RUN_NAME,
    run_output_dir=RUN_OUTPUT_DIR,
    run_status_path=RUN_STATUS_PATH,
    run_progress_path=RUN_PROGRESS_PATH,
    run_heartbeat_path=RUN_HEARTBEAT_PATH,
    run_failure_log_path=RUN_FAILURE_LOG_PATH,
    model_registry=MODEL_REGISTRY,
    run_model_keys=RUN_MODEL_KEYS,
    benchmark_seeds=BENCHMARK_SEEDS,
    loss_keys_by_model=LOSS_KEYS_BY_MODEL,
    fixed_loss_key=FIXED_LOSS_KEY,
    checkpoint_selection_rule=CHECKPOINT_SELECTION_RULE,
    deterministic_mode=DETERMINISTIC_MODE,
    resume_if_available=RESUME_IF_AVAILABLE,
    skip_completed_seeds=SKIP_COMPLETED_SEEDS,
    force_rerun_seeds=FORCE_RERUN_SEEDS,
    allow_partial_aggregation=ALLOW_PARTIAL_AGGREGATION,
    n_epochs=N_EPOCHS,
    early_stop_patience=EARLY_STOP_PATIENCE,
    max_grad_norm=MAX_GRAD_NORM,
    log_every_steps=LOG_EVERY_STEPS,
    heartbeat_every_steps=HEARTBEAT_EVERY_STEPS,
    ece_n_bins=ECE_N_BINS,
    confidence_thresholds=CONFIDENCE_THRESHOLDS,
    dataloader_num_workers=DATALOADER_NUM_WORKERS,
    dataloader_prefetch_factor=DATALOADER_PREFETCH_FACTOR,
    enable_split_hygiene_recompute=ENABLE_SPLIT_HYGIENE_RECOMPUTE,
    enable_truncation_evidence=ENABLE_TRUNCATION_EVIDENCE,
    enable_calibration=ENABLE_CALIBRATION,
    enable_threshold_security_artifacts=ENABLE_THRESHOLD_SECURITY_ARTIFACTS,
    enable_reliability_diagrams=ENABLE_RELIABILITY_DIAGRAMS,
    enable_latency_benchmark=ENABLE_LATENCY_BENCHMARK,
    generate_heavy_artifacts_during_training=GENERATE_HEAVY_ARTIFACTS_DURING_TRAINING,
    generate_heavy_artifacts_after_training=GENERATE_HEAVY_ARTIFACTS_AFTER_TRAINING,
    latency_protocol=LATENCY_PROTOCOL,
    split_summaries=SPLIT_SUMMARIES,
    split_hygiene_evidence=SPLIT_HYGIENE_EVIDENCE,
    device=DEVICE,
    cuda_bf16=CUDA_BF16,
)

RUNNER = FinalConfirmatoryRunner(RUNNER_CONTEXT)

print("Module-backed confirmatory runner context is ready.")

Module-backed confirmatory runner context is ready.


In [13]:
seed_everything = RUNNER.seed_everything
write_run_status = RUNNER.write_run_status
write_run_progress = RUNNER.write_run_progress
write_run_heartbeat = RUNNER.write_run_heartbeat

build_pending_work_plan = RUNNER.build_pending_work_plan
count_completed_seed_runs = RUNNER.count_completed_seed_runs
count_completed_models = RUNNER.count_completed_models

run_confirmatory_model = RUNNER.run_confirmatory_model
rebuild_run_aggregates = RUNNER.rebuild_run_aggregates

print("Core training, checkpoint, resume, failure-isolation, and aggregation utilities are ready.")

Core training, checkpoint, resume, failure-isolation, and aggregation utilities are ready.


In [14]:
seed_everything(BENCHMARK_SEEDS[0], deterministic=DETERMINISTIC_MODE)

if RUN_FAILURE_LOG_PATH.exists():
    loaded_failures = load_json(RUN_FAILURE_LOG_PATH)
    RUN_FAILURES = loaded_failures if isinstance(loaded_failures, list) else []
else:
    RUN_FAILURES = []

MODEL_RUN_TABLES = {}
MODEL_TRUNCATION_OVERVIEW = {}

RUNNER.set_runtime_state(
    run_failures=RUN_FAILURES,
    model_run_tables=MODEL_RUN_TABLES,
    model_truncation_overview=MODEL_TRUNCATION_OVERVIEW,
)

RUN_FAILURES = RUNNER.run_failures
MODEL_RUN_TABLES = RUNNER.model_run_tables
MODEL_TRUNCATION_OVERVIEW = RUNNER.model_truncation_overview

PENDING_WORK_PLAN = build_pending_work_plan()
save_csv(PENDING_WORK_PLAN, RUN_OUTPUT_DIR / "pending_work_plan.csv", index=False)

completed_rows = int((PENDING_WORK_PLAN["state"] == "completed").sum()) if not PENDING_WORK_PLAN.empty else 0
resumable_rows = int((PENDING_WORK_PLAN["state"] == "resumable").sum()) if not PENDING_WORK_PLAN.empty else 0
failed_rows = int((PENDING_WORK_PLAN["state"] == "failed_last_attempt").sum()) if not PENDING_WORK_PLAN.empty else 0
pending_rows = int((PENDING_WORK_PLAN["state"] == "pending").sum()) if not PENDING_WORK_PLAN.empty else 0

print("Pending-work planner summary:")
print(f"- completed           : {completed_rows}")
print(f"- resumable           : {resumable_rows}")
print(f"- failed last attempt : {failed_rows}")
print(f"- pending fresh       : {pending_rows}")
print(f"- planner csv         : {RUN_OUTPUT_DIR / 'pending_work_plan.csv'}")

display(PENDING_WORK_PLAN)

write_run_status(
    "ready",
    "run_state_planner",
    completed_rows=int(completed_rows),
    resumable_rows=int(resumable_rows),
    failed_rows=int(failed_rows),
    pending_rows=int(pending_rows),
)
write_run_progress(
    stage="run_state_planner",
    completed_models=int(count_completed_models()),
    completed_seeds=int(count_completed_seed_runs()),
)
write_run_heartbeat(
    "run_state_planner_ready",
    completed_rows=int(completed_rows),
    resumable_rows=int(resumable_rows),
    failed_rows=int(failed_rows),
    pending_rows=int(pending_rows),
)

print("Run-state discovery initialized. Execute model cells independently in any order.")

Pending-work planner summary:
- completed           : 0
- resumable           : 0
- failed last attempt : 0
- pending fresh       : 9
- planner csv         : C:\Users\Froi\Documents\project\injection-alert-system\results\v3_907k_cleaned_final_confirmatory_weighted_ce_3seed_20260412_035441\pending_work_plan.csv


,model_key,loss_key,seed,state
0,distilbert,weighted_ce,42,pending
1,distilbert,weighted_ce,1337,pending
2,distilbert,weighted_ce,2026,pending
3,minilm_l6,weighted_ce,42,pending
4,minilm_l6,weighted_ce,1337,pending
5,minilm_l6,weighted_ce,2026,pending
6,tinybert_bigru_attn,weighted_ce,42,pending
7,tinybert_bigru_attn,weighted_ce,1337,pending
8,tinybert_bigru_attn,weighted_ce,2026,pending


Run-state discovery initialized. Execute model cells independently in any order.


In [19]:
# Model run 1/3: distilbert
run_confirmatory_model("distilbert")


MODEL RUN START | model=distilbert | losses=['weighted_ce'] | seeds=[42, 1337, 2026]
Using cached resources for model=distilbert

Loss variant start | model=distilbert | loss=weighted_ce
[skip] model=distilbert | loss=weighted_ce | seed=42 already completed
[skip] model=distilbert | loss=weighted_ce | seed=1337 already completed
[skip] model=distilbert | loss=weighted_ce | seed=2026 already completed
MODEL RUN END | model=distilbert | completed_rows=1


,model_key,loss_key,architecture,architecture_family,head_type,experiment_phase,fixed_loss_key,n_seeds,best_epoch_mean,best_epoch_std,...,model_size_mb_ci95_lower,model_size_mb_ci95_upper,training_workflow_runtime_sec_mean,training_workflow_runtime_sec_std,training_workflow_runtime_sec_ci95_lower,training_workflow_runtime_sec_ci95_upper,mean_epoch_training_time_sec_mean,mean_epoch_training_time_sec_std,mean_epoch_training_time_sec_ci95_lower,mean_epoch_training_time_sec_ci95_upper
0,distilbert,weighted_ce,transformer,backbone_with_standard_head,mean_pool_mlp,controlled_backbone_benchmark,weighted_ce,3,3.333333,0.57735,...,253.911148,253.911148,1830.03327,4.45691,1824.989802,1835.076738,431.329788,1.046024,430.1461,432.513476


In [16]:
# Model run 2/3: minilm_l6
run_confirmatory_model("minilm_l6")


MODEL RUN START | model=minilm_l6 | losses=['weighted_ce'] | seeds=[42, 1337, 2026]
Preparing tokenizer/resources for model=minilm_l6 (nreimers/MiniLM-L6-H384-uncased)
Tokenization started for train/validation/test splits ...
Tokenization finished | samples=199,039 | elapsed=7.2s | throughput=27518.3 samples/s
Resources ready for model=minilm_l6 | elapsed=8.4s | cache_size=2

Loss variant start | model=minilm_l6 | loss=weighted_ce
[seed-start] model=minilm_l6 | loss=weighted_ce | seed=42 | train_batches=1,250 | val_batches=77 | test_batches=77


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: nreimers/MiniLM-L6-H384-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Epoch 1/4 | model=minilm_l6 | seed=42


minilm_l6|seed42|epoch1:   0%|          | 0/1250 [00:00<?, ?it/s]

...

minilm_l6|seed2026|epoch3:   0%|          | 0/1250 [00:00<?, ?it/s]

  step   200/1,250 | loss=0.0197 | lr=9.19e-06 | speed=7.30 steps/s | eta=143.8s
  step   400/1,250 | loss=0.0215 | lr=7.91e-06 | speed=7.31 steps/s | eta=116.3s
  step   600/1,250 | loss=0.0226 | lr=6.66e-06 | speed=7.30 steps/s | eta=89.0s
  step   800/1,250 | loss=0.0226 | lr=5.47e-06 | speed=7.30 steps/s | eta=61.7s
  step 1,000/1,250 | loss=0.0224 | lr=4.36e-06 | speed=7.30 steps/s | eta=34.2s
  step 1,200/1,250 | loss=0.0223 | lr=3.34e-06 | speed=7.30 steps/s | eta=6.9s
  step 1,250/1,250 | loss=0.0221 | lr=3.10e-06 | speed=7.30 steps/s | eta=0.0s
  best checkpoint saved: best_minilm_l6_weighted_ce_seed2026.pt
  last checkpoint saved: last_minilm_l6_weighted_ce_seed2026.pt
  epoch-summary | model=minilm_l6 | seed=2026 | epoch=3/4 | train_loss=0.0221 | val_loss=0.0239 | val_macro_f1=0.9921 | lr=3.10e-06 | epoch_time=171.2s | elapsed=538.4s | state=BEST

Epoch 4/4 | model=minilm_l6 | seed=2026


minilm_l6|seed2026|epoch4:   0%|          | 0/1250 [00:00<?, ?it/s]

  step   200/1,250 | loss=0.0214 | lr=2.23e-06 | speed=7.29 steps/s | eta=144.1s
  step   400/1,250 | loss=0.0199 | lr=1.48e-06 | speed=7.29 steps/s | eta=116.6s
  step   600/1,250 | loss=0.0199 | lr=8.73e-07 | speed=7.29 steps/s | eta=89.1s
  step   800/1,250 | loss=0.0194 | lr=4.22e-07 | speed=7.29 steps/s | eta=61.7s
  step 1,000/1,250 | loss=0.0188 | lr=1.31e-07 | speed=7.29 steps/s | eta=34.3s
  step 1,200/1,250 | loss=0.0186 | lr=5.24e-09 | speed=7.29 steps/s | eta=6.9s
  step 1,250/1,250 | loss=0.0184 | lr=0.00e+00 | speed=7.30 steps/s | eta=0.0s
  best checkpoint saved: best_minilm_l6_weighted_ce_seed2026.pt
  last checkpoint saved: last_minilm_l6_weighted_ce_seed2026.pt
  epoch-summary | model=minilm_l6 | seed=2026 | epoch=4/4 | train_loss=0.0184 | val_loss=0.0231 | val_macro_f1=0.9925 | lr=0.00e+00 | epoch_time=171.3s | elapsed=717.9s | state=BEST
[seed-complete] model=minilm_l6 | loss=weighted_ce | seed=2026 | best_epoch=4 | val_macro_f1=0.9925 | test_macro_f1=0.9883 | runti

,model_key,loss_key,architecture,architecture_family,head_type,experiment_phase,fixed_loss_key,n_seeds,best_epoch_mean,best_epoch_std,...,model_size_mb_ci95_lower,model_size_mb_ci95_upper,training_workflow_runtime_sec_mean,training_workflow_runtime_sec_std,training_workflow_runtime_sec_ci95_lower,training_workflow_runtime_sec_ci95_upper,mean_epoch_training_time_sec_mean,mean_epoch_training_time_sec_std,mean_epoch_training_time_sec_ci95_lower,mean_epoch_training_time_sec_ci95_upper
0,minilm_l6,weighted_ce,transformer,backbone_with_standard_head,mean_pool_mlp,controlled_backbone_benchmark,weighted_ce,3,3.666667,0.57735,...,87.025894,87.025894,732.261834,0.740984,731.423332,733.100337,171.167074,0.148486,170.999047,171.335102


In [17]:
# Model run 3/3: tinybert_bigru_attn
run_confirmatory_model("tinybert_bigru_attn")


MODEL RUN START | model=tinybert_bigru_attn | losses=['weighted_ce'] | seeds=[42, 1337, 2026]
Preparing tokenizer/resources for model=tinybert_bigru_attn (huawei-noah/TinyBERT_General_6L_768D)
Tokenization started for train/validation/test splits ...
Tokenization finished | samples=199,039 | elapsed=8.0s | throughput=24827.2 samples/s
Resources ready for model=tinybert_bigru_attn | elapsed=9.9s | cache_size=3

Loss variant start | model=tinybert_bigru_attn | loss=weighted_ce
[seed-start] model=tinybert_bigru_attn | loss=weighted_ce | seed=42 | train_batches=2,499 | val_batches=154 | test_batches=153


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: huawei-noah/TinyBERT_General_6L_768D
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
fit_denses.{0, 1, 2, 3, 4, 5, 6}.bias      | UNEXPECTED |  | 
fit_denses.{0, 1, 2, 3, 4, 5, 6}.weight    | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Epoch 1/4 | model=tinybert_bigru_attn | seed=42


tinybert_bigru_attn|seed42|epoch1:   0%|          | 0/2499 [00:00<?, ?it/s]

...

tinybert_bigru_attn|seed2026|epoch3:   0%|          | 0/2499 [00:00<?, ?it/s]

  step   200/2,499 | loss=0.0147 | lr=1.50e-05 | speed=5.05 steps/s | eta=455.0s
  step   400/2,499 | loss=0.0133 | lr=1.40e-05 | speed=5.05 steps/s | eta=415.3s
  step   600/2,499 | loss=0.0132 | lr=1.30e-05 | speed=5.06 steps/s | eta=375.5s
  step   800/2,499 | loss=0.0146 | lr=1.21e-05 | speed=5.06 steps/s | eta=335.9s
  step 1,000/2,499 | loss=0.0144 | lr=1.11e-05 | speed=5.06 steps/s | eta=296.5s
  step 1,200/2,499 | loss=0.0152 | lr=1.02e-05 | speed=5.05 steps/s | eta=257.0s
  step 1,400/2,499 | loss=0.0151 | lr=9.26e-06 | speed=5.06 steps/s | eta=217.3s
  step 1,600/2,499 | loss=0.0153 | lr=8.37e-06 | speed=5.06 steps/s | eta=177.8s
  step 1,800/2,499 | loss=0.0152 | lr=7.50e-06 | speed=5.06 steps/s | eta=138.2s
  step 2,000/2,499 | loss=0.0152 | lr=6.67e-06 | speed=5.06 steps/s | eta=98.7s
  step 2,200/2,499 | loss=0.0152 | lr=5.87e-06 | speed=5.06 steps/s | eta=59.1s
  step 2,400/2,499 | loss=0.0153 | lr=5.11e-06 | speed=5.06 steps/s | eta=19.6s
  step 2,499/2,499 | loss=0.015

tinybert_bigru_attn|seed2026|epoch4:   0%|          | 0/2499 [00:00<?, ?it/s]

  step   200/2,499 | loss=0.0141 | lr=4.05e-06 | speed=5.06 steps/s | eta=454.6s
  step   400/2,499 | loss=0.0145 | lr=3.40e-06 | speed=5.06 steps/s | eta=414.9s
  step   600/2,499 | loss=0.0143 | lr=2.81e-06 | speed=5.05 steps/s | eta=375.7s
  step   800/2,499 | loss=0.0137 | lr=2.26e-06 | speed=5.06 steps/s | eta=335.7s
  step 1,000/2,499 | loss=0.0138 | lr=1.77e-06 | speed=5.06 steps/s | eta=296.2s
  step 1,200/2,499 | loss=0.0135 | lr=1.34e-06 | speed=5.06 steps/s | eta=256.6s
  step 1,400/2,499 | loss=0.0134 | lr=9.61e-07 | speed=5.06 steps/s | eta=217.3s
  step 1,600/2,499 | loss=0.0132 | lr=6.46e-07 | speed=5.06 steps/s | eta=177.8s
  step 1,800/2,499 | loss=0.0131 | lr=3.92e-07 | speed=5.06 steps/s | eta=138.3s
  step 2,000/2,499 | loss=0.0128 | lr=2.00e-07 | speed=5.06 steps/s | eta=98.7s
  step 2,200/2,499 | loss=0.0128 | lr=7.22e-08 | speed=5.06 steps/s | eta=59.1s
  step 2,400/2,499 | loss=0.0127 | lr=8.03e-09 | speed=5.06 steps/s | eta=19.6s
  step 2,499/2,499 | loss=0.012

,model_key,loss_key,architecture,architecture_family,head_type,experiment_phase,fixed_loss_key,n_seeds,best_epoch_mean,best_epoch_std,...,model_size_mb_ci95_lower,model_size_mb_ci95_upper,training_workflow_runtime_sec_mean,training_workflow_runtime_sec_std,training_workflow_runtime_sec_ci95_lower,training_workflow_runtime_sec_ci95_upper,mean_epoch_training_time_sec_mean,mean_epoch_training_time_sec_std,mean_epoch_training_time_sec_ci95_lower,mean_epoch_training_time_sec_ci95_upper
0,tinybert_bigru_attn,weighted_ce,tinybert_bigru_attention,architecture_search_variant,bigru_attention_mlp,architecture_search,weighted_ce,3,4.0,0.0,...,262.182632,262.182632,2094.657296,0.943142,2093.590031,2095.724562,494.417992,0.226351,494.161853,494.674132


In [20]:
all_loss_df, model_df, run_manifest = rebuild_run_aggregates(
    generate_deferred_heavy_artifacts=GENERATE_HEAVY_ARTIFACTS_AFTER_TRAINING
)

if all_loss_df.empty:
    print("No completed model/loss aggregate rows are available yet.")
    print("Run one or more model cells, then rerun this aggregation cell.")
else:
    summary_cols = [
        "model_key",
        "loss_key",
        "n_seeds",
        "val_macro_f1_mean",
        "test_macro_f1_mean",
        "test_accuracy_mean",
        "training_workflow_runtime_sec_mean",
        "mean_epoch_training_time_sec_mean",
    ]
    display_cols = [col for col in summary_cols if col in all_loss_df.columns]

    print("Top model/loss aggregates:")
    display(all_loss_df[display_cols].head(20))

    if not model_df.empty:
        model_display_cols = [col for col in summary_cols if col in model_df.columns]
        print("\nPer-model benchmark summary (best loss variant per model):")
        display(model_df[model_display_cols])

print(f"Saved all-loss aggregates : {RUN_OUTPUT_DIR / 'all_loss_variant_aggregates.csv'}")
print(f"Saved model summary       : {RUN_OUTPUT_DIR / 'model_benchmark_summary.csv'}")
print(f"Saved run manifest        : {RUN_OUTPUT_DIR / 'run_manifest.json'}")
print(f"Run status path           : {RUN_STATUS_PATH}")
print(f"Run progress path         : {RUN_PROGRESS_PATH}")
print(f"Run heartbeat path        : {RUN_HEARTBEAT_PATH}")
print(f"Run failure log path      : {RUN_FAILURE_LOG_PATH}")

Using cached resources for model=distilbert
Using cached resources for model=minilm_l6
Using cached resources for model=tinybert_bigru_attn
Top model/loss aggregates:


,model_key,loss_key,n_seeds,val_macro_f1_mean,test_macro_f1_mean,test_accuracy_mean,training_workflow_runtime_sec_mean,mean_epoch_training_time_sec_mean
0,minilm_l6,weighted_ce,3,0.992468,0.990434,0.992429,732.261834,171.167074
1,distilbert,weighted_ce,3,0.993857,0.989142,0.992754,1830.033270,431.329788
2,tinybert_bigru_attn,weighted_ce,3,0.993755,0.988904,0.992805,2094.657296,494.417992



Per-model benchmark summary (best loss variant per model):


,model_key,loss_key,n_seeds,val_macro_f1_mean,test_macro_f1_mean,test_accuracy_mean,training_workflow_runtime_sec_mean,mean_epoch_training_time_sec_mean
0,minilm_l6,weighted_ce,3,0.992468,0.990434,0.992429,732.261834,171.167074
1,distilbert,weighted_ce,3,0.993857,0.989142,0.992754,1830.033270,431.329788
2,tinybert_bigru_attn,weighted_ce,3,0.993755,0.988904,0.992805,2094.657296,494.417992


Saved all-loss aggregates : C:\Users\Froi\Documents\project\injection-alert-system\results\v3_907k_cleaned_final_confirmatory_weighted_ce_3seed_20260412_035441\all_loss_variant_aggregates.csv
Saved model summary       : C:\Users\Froi\Documents\project\injection-alert-system\results\v3_907k_cleaned_final_confirmatory_weighted_ce_3seed_20260412_035441\model_benchmark_summary.csv
Saved run manifest        : C:\Users\Froi\Documents\project\injection-alert-system\results\v3_907k_cleaned_final_confirmatory_weighted_ce_3seed_20260412_035441\run_manifest.json
Run status path           : C:\Users\Froi\Documents\project\injection-alert-system\results\v3_907k_cleaned_final_confirmatory_weighted_ce_3seed_20260412_035441\run_status.json
Run progress path         : C:\Users\Froi\Documents\project\injection-alert-system\results\v3_907k_cleaned_final_confirmatory_weighted_ce_3seed_20260412_035441\run_progress.json
Run heartbeat path        : C:\Users\Froi\Documents\project\injection-alert-system\resul